# Brno fine-tuning worker

Runs `scripts/colab_worker.py`: works through `experiments/queue.json` on the `video-3d-bbox` branch
(prepare Brno recordings, fine-tuning experiments), pulling the branch between jobs so new jobs and code
are picked up without touching this notebook. Progress: `Final-Project-CHULA/runs/worker/` (per-job logs,
`heartbeat.json`) and `Final-Project-CHULA/runs/experiments/leaderboard.csv`.

Use a GPU runtime, then Run all and leave the tab open.

**No sign-in popup (optional):** run `rclone config` once on your own machine (a Google Drive remote named
`gdrive`), then add the contents of `~/.config/rclone/rclone.conf` as a Colab secret named `RCLONE_CONF`
with notebook access on. The next cell then mounts Drive with rclone at the same path; without the secret it
falls back to `drive.mount`. The secret gives full access to your Drive: don't share the notebook with it.

In [ ]:
BRANCH = 'video-3d-bbox'
# Drive at /content/drive/MyDrive: with the RCLONE_CONF secret, via rclone (no sign-in popup);
# otherwise the usual drive.mount.
import os, subprocess
from google.colab import userdata
try:
    rclone_conf = userdata.get('RCLONE_CONF')
except Exception:  # secret missing or notebook access not granted
    rclone_conf = None
if rclone_conf:
    os.makedirs('/root/.config/rclone', exist_ok=True)
    with open('/root/.config/rclone/rclone.conf', 'w') as f:
        f.write(rclone_conf)
    # install.sh exits non-zero when rclone is already installed, so no check here
    subprocess.run('curl -s https://rclone.org/install.sh | bash', shell=True, stdout=subprocess.DEVNULL)
    subprocess.run('apt-get -qq install -y fuse3', shell=True, check=True, stdout=subprocess.DEVNULL)
    os.makedirs('/content/drive/MyDrive', exist_ok=True)
    subprocess.run(['rclone', 'mount', 'gdrive:', '/content/drive/MyDrive', '--daemon', '--vfs-cache-mode', 'writes'],
                   check=True)
    print('Drive mounted with rclone')
else:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
PROJECT = next(c for c in ['/content/drive/MyDrive/Final-Project-CHULA', '/content/drive/Shareddrives/Final-Project-CHULA']
               if os.path.isdir(c))
BRNO = f'{PROJECT}/Brno/data/2016-ITS-BrnoCompSpeed'
assert os.path.isdir(f'{BRNO}/dataset'), BRNO
print(PROJECT, BRNO, sep='\n')

In [ ]:
%cd /content
!rm -rf capcapcar-speed && git clone -q -b {BRANCH} https://github.com/WalkerCze3/capcapcar-speed.git
%cd capcapcar-speed
!pip install -q -r requirements-video.txt

In [ ]:
import subprocess
p = subprocess.Popen(['python', 'scripts/colab_worker.py', '--project', PROJECT, '--brno', BRNO, '--branch', BRANCH],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in p.stdout:
    print(line, end='', flush=True)